<a href="https://colab.research.google.com/github/marielary/drought-analysis-haute-matsiatra/blob/main/Pipeline_Data_for_VAR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install rioxarray -q


In [ ]:
!earthengine authenticate

In [ ]:
"""
================================================================================
PIPELINE AGROCLIMATIQUE COMPLET — HAUTE MATSIATRA, MADAGASCAR (2012–2024)
CULTURES : RIZ, MAÏS, ARACHIDE

  ✅ Altitude variable par district via SRTM (cohérence avec pipeline SPI/SPEI)

Prérequis :
  → Avoir exécuté pretraitement_spi_spei_final_v2.py au préalable
  → Fichiers requis dans SPI_SPEI_FINAL/ :
       all_indices_saison.pkl
================================================================================
"""

In [ ]:
# ============================================================================
# 0. PACKAGES
# ============================================================================

import os, gc, shutil, gzip, zipfile, warnings, time, pickle
import requests
import numpy as np
import pandas as pd
import geopandas as gpd
from datetime import datetime, date, timedelta
from io import BytesIO

warnings.filterwarnings('ignore')

def install_if_missing(pkg):
    try: __import__(pkg)
    except ImportError: os.system(f"pip install {pkg} -q")

for pkg in ['cdsapi', 'xarray', 'netCDF4', 'rasterio', 'earthengine-api']:
    install_if_missing(pkg)

import xarray as xr
import rasterio
from rasterio.mask import mask as rasterio_mask

print("="*80)
print("PIPELINE AGROCLIMATIQUE — HAUTE MATSIATRA  [v2]")
print("SPI/SPEI chargés depuis fichiers précalculés")
print("="*80)
print(f"Démarrage : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# ============================================================================
# 1. CONFIGURATION
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

BASE_DRIVE = "/content/drive/MyDrive/DATA_THESES_MADATLAS"

SHAPEFILE_DISTRICTS = "/content/region_district_HM.shp"
SHAPEFILE_SOLS      = "/content/Soil_map_HM_VF.shp"
EXCEL_RENDEMENTS    = "/content/rendemenrt_haute_matsiatra.xlsx"

OUTPUT_DIR          = f"{BASE_DRIVE}/DONNEES_COMPLETES_"
ECMWF_API_KEY       = "ff131d43-593c-42ae-b827-b2fac32e75c8"

# ── Chemins fichiers SPI/SPEI précalculés ────────────────────────────────────
# Générés par pretraitement_spi_spei_final_v2.py
SPEI_PKL = f"{BASE_DRIVE}/BASELINE/SPI_SPEI_FINAL/all_indices_saison.pkl"
SPEI_CSV = f"{BASE_DRIVE}/BASELINE/SPI_SPEI_FINAL/indices_saison_tous_districts.csv"

ANNEES       = range(2012, 2019)   # campagnes 2011/12 → 2017/18
COL_DISTRICT = "ADM2_EN"

# Phases culturales (campagne Nov–Mai)
PHASES_MOIS = {
    'P1': [(11, 'Nov', -1), (12, 'Dec', -1)],
    'P2': [(1,  'Jan',  0), (2,  'Feb',  0)],
    'P3': [(3,  'Mar',  0), (4,  'Apr',  0), (5, 'May', 0)],
}
PHASES = list(PHASES_MOIS.keys())

# Paramètres réseau
BLOC_ERA5_JOURS = 10
DELAI_ERA5_S    = 12
DELAI_CHIRPS_S  = 2
MAX_RETRY       = 3

In [ ]:
# ============================================================================
# 2. CHARGEMENT DES INDICES SPEI PRÉCALCULÉS
# ============================================================================

print("\n1. CHARGEMENT DES INDICES SPEI PRÉCALCULÉS")
print("-"*50)

def charger_spei_precompute():
    """
    Charge le tableau des indices SPEI saisonniers pré-calculés.

    Colonnes attendues (générées par pretraitement_spi_spei_final_v2.py) :
      district        : nom du district
      annee_campagne  : année de début de campagne (ex: 2012 = campagne 2012/13)
      SPEI3_fev       : SPEI-3 de février  (fenêtre Déc–Jan–Fév)
      SPEI3_fev_cat   : catégorie (Normal / Modérée / Sévère / Extrême)
      SPEI6_mai       : SPEI-6 de mai      (bilan saison Déc–Mai)
      SPEI6_mai_cat   : catégorie
      SPEI12_mai      : SPEI-12 de mai     (fond hydrologique)
      SPEI12_mai_cat  : catégorie
    """
    # Priorité 1 : pickle (plus rapide)
    if os.path.exists(SPEI_PKL):
        try:
            with open(SPEI_PKL, 'rb') as f:
                raw = pickle.load(f)
            # raw est un dict {district: DataFrame}
            dfs = []
            for district, df in raw.items():
                df = df.copy()
                df.insert(0, 'district', district)
                dfs.append(df)
            df_spei = pd.concat(dfs, ignore_index=True)
            print(f"  ✓ Chargé depuis PKL : {SPEI_PKL}")
            print(f"    {len(df_spei)} lignes × {len(df_spei.columns)} colonnes")
            return df_spei
        except Exception as e:
            print(f" PKL illisible ({e}) → tentative CSV")

    # Priorité 2 : CSV consolidé
    if os.path.exists(SPEI_CSV):
        try:
            df_spei = pd.read_csv(SPEI_CSV)
            df_spei['annee_campagne'] = df_spei['annee_campagne'].astype(int)
            print(f"  ✓ Chargé depuis CSV : {SPEI_CSV}")
            return df_spei
        except Exception as e:
            print(f"CSV illisible ({e})")

    # Échec
    print(f"\n INDICES SPEI INTROUVABLES !")
    print(f"   Fichiers cherchés :")
    print(f"     • {SPEI_PKL}")
    print(f"     • {SPEI_CSV}")
    print(f"\n   → Exécutez d'abord : pretraitement_spi_spei_final_v2.py")
    raise FileNotFoundError("Indices SPEI précalculés non trouvés.")

df_spei_ref = charger_spei_precompute()

# Vérification des colonnes attendues
COLS_SPEI_ATTENDUES = ['SPEI3_fev', 'SPEI6_mai', 'SPEI12_mai']
manquantes = [c for c in COLS_SPEI_ATTENDUES if c not in df_spei_ref.columns]
if manquantes:
    print(f" Colonnes SPEI manquantes : {manquantes}")
    print(f"     Colonnes disponibles : {list(df_spei_ref.columns)}")
else:
    print(f"  ✓ Colonnes SPEI vérifiées : {COLS_SPEI_ATTENDUES}")

# Aperçu
print(f"\n  Aperçu des indices SPEI précalculés :")
print(f"  Districts  : {sorted(df_spei_ref['district'].unique())}")
print(f"  Campagnes  : {sorted(int(x) for x in df_spei_ref['annee_campagne'].unique())}")
print()


def get_spei_district_annee(district, annee_campagne):
    """
    Retourne un dict avec les 3 indices SPEI pour un district/campagne.
    annee_campagne = année de début (ex: 2012 pour campagne 2012/13).
    Retourne NaN si non disponible (campagne hors baseline 1991-2020).
    """
    mask = (
        (df_spei_ref['district'] == district) &
        (df_spei_ref['annee_campagne'].astype(int) == int(annee_campagne))  # cast np.int64 → int
    )
    if not mask.any():
        return {
            'spei3_fev': np.nan, 'spei3_fev_cat': 'NA',
            'spei6_mai': np.nan, 'spei6_mai_cat': 'NA',
            'spei12_mai': np.nan, 'spei12_mai_cat': 'NA',
        }
    row = df_spei_ref.loc[mask].iloc[0]
    return {
        'spei3_fev'   : float(row.get('SPEI3_fev',  np.nan)),
        'spei3_fev_cat': str(row.get('SPEI3_fev_cat', 'NA')),
        'spei6_mai'   : float(row.get('SPEI6_mai',  np.nan)),
        'spei6_mai_cat': str(row.get('SPEI6_mai_cat', 'NA')),
        'spei12_mai'  : float(row.get('SPEI12_mai', np.nan)),
        'spei12_mai_cat': str(row.get('SPEI12_mai_cat', 'NA')),
    }


In [ ]:
# ============================================================================
# 3. PARAMÈTRES SCIENTIFIQUES
# ============================================================================

# Constantes FAO-56
ALBEDO, SIGMA, G = 0.23, 4.903e-9, 0.0
P0, T0_K, L_ad   = 101.3, 293.15, 0.0065

# GDD (McMaster & Wilhelm 1997)
GDD_PARAMS = {
    'Rice':      {'T_base': 10.0, 'T_ceil': 40.0},
    'Maize':     {'T_base':  8.0, 'T_ceil': 44.0},
    'Groundnut': {'T_base': 12.0, 'T_ceil': 40.0},
}

# Kc par phase (FAO-33)
KC_PARAMS = {
    'Rice':      {'P1': 1.05, 'P2': 1.20, 'P3': 0.90},
    'Maize':     {'P1': 0.70, 'P2': 1.15, 'P3': 0.60},
    'Groundnut': {'P1': 0.60, 'P2': 1.05, 'P3': 0.75},
}

ROOT_DEPTH  = {'Rice': 500, 'Maize': 600, 'Groundnut': 500}
DEPLETION_P = {'Rice': 0.20, 'Maize': 0.55, 'Groundnut': 0.50}

# FAO_SOIL_AWC = {
#     'Fo': 100, 'Fo1': 100, 'Fo2': 105, 'Fo3': 110,
#     'Fo83': 108, 'Fo83-2b': 108,
#     'Ao': 115, 'Af': 110, 'Ag': 120,
#     'Lc': 140, 'Lo': 130, 'Lk': 135,
#     'Gd': 180, 'Ge': 175, 'Gh': 170,
#     'Jc': 200, 'Je': 195, 'Jd': 190,
#     'To': 150, 'Th': 155,
#     'Vp': 160, 'Vc': 165,
#     'Bc': 130, 'Bd': 125,
#     'Nd': 120, 'default': 120,
# }
FAO_SOIL_AWC = {
    # Ferralsols
    'Fo': 100, 'Fo1': 100, 'Fo2': 105, 'Fo3': 110,
    'Fo81': 103, 'Fo81-2b': 103,   # ← ajout
    'Fo82': 105, 'Fo82-2b': 105,   # ← ajout
    'Fo83': 108, 'Fo83-2b': 108,
    # Cambisols
    'Bc': 130, 'Bd': 125, 'Bd33': 125, 'Bd33-2a': 125,
    'Bf': 100, 'Bf11': 100, 'Bf11-2c': 100,   # ← ajout
    # Luvisols
    'Lc': 140, 'Lo': 130, 'Lk': 135,
    'Lf': 135, 'Lf85': 135, 'Lf85-2ab': 135,  # ← ajout
    # Nitosols
    'Ne': 155, 'Ne47': 155, 'Ne47-2b': 155,   # ← ajout
    # Regosols
    'Rd': 85, 'Rd22': 85, 'Rd22-2c': 85,      # ← ajout
    # Autres
    'Ao': 115, 'Af': 110, 'Ag': 120,
    'Gd': 180, 'Ge': 175, 'Gh': 170,
    'Jc': 200, 'Je': 195, 'Jd': 190,
    'To': 150, 'Th': 155,
    'Vp': 160, 'Vc': 165,
    'Nd': 120, 'default': 120,
}

def get_awc(code):
    if not code or pd.isna(code): return FAO_SOIL_AWC['default']
    c = str(code).strip()
    return (FAO_SOIL_AWC.get(c) or FAO_SOIL_AWC.get(c[:4])
            or FAO_SOIL_AWC.get(c[:2]) or FAO_SOIL_AWC['default'])

SEUIL_PLUIE   = 1.0
SEUIL_EXTREME = 50.0
SEUIL_TSTRESS = 35.0
SEUIL_HOT_N   = 25.0
ET0_CESS      = 5.0
SEUIL_SW_CESS = 5.0
DUR_CESS      = 3

In [ ]:
# ============================================================================
# 4. DOSSIERS
# ============================================================================

TEMP_DIR = "/tmp/agro_hm"
os.makedirs(TEMP_DIR, exist_ok=True)
for d in [f"{OUTPUT_DIR}/02_PROCESSED/ANNUAL", f"{OUTPUT_DIR}/03_FINAL"]:
    os.makedirs(d, exist_ok=True)
print(f"\n2. DOSSIERS")
print(f"   Résultats : {OUTPUT_DIR}")
print(f"   Temp      : {TEMP_DIR}")

In [ ]:
# ============================================================================
# 5. APIs
# ============================================================================

print("\n3. INITIALISATION APIs")
import cdsapi, ee

with open(os.path.expanduser("~/.cdsapirc"), 'w') as f:
    f.write(f"url: https://cds.climate.copernicus.eu/api\nkey: {ECMWF_API_KEY}\n")
cds = cdsapi.Client()
print("   ✓ Client CDS")

try:
    ee.Initialize()
    print("   ✓ Earth Engine")
except:
    ee.Authenticate()
    ee.Initialize()


In [ ]:
# ============================================================================
# 6. DONNÉES SPATIALES
# ============================================================================

print("\n4. DONNÉES SPATIALES")
print("-"*40)

gdf = gpd.read_file(SHAPEFILE_DISTRICTS)
for c in ['ADM2_EN','ADM2_FR','NAME','NOM','DISTRICT']:
    if c in gdf.columns: COL_DISTRICT = c; break

if gdf.crs is None:          gdf = gdf.set_crs("EPSG:4326")
elif gdf.crs.to_epsg() != 4326: gdf = gdf.to_crs("EPSG:4326")

districts = gdf[COL_DISTRICT].tolist()
bounds    = gdf.total_bounds
bbox_cds  = [bounds[3], bounds[0], bounds[1], bounds[2]]
print(f"   {len(districts)} districts : {districts}")

# Altitude variable par district via SRTM (cohérence avec pipeline SPI/SPEI)
alt_districts = {}
print("   Altitude SRTM par district...", end='', flush=True)
for _, row in gdf.iterrows():
    dist = row[COL_DISTRICT]
    try:
        coords  = list(row.geometry.exterior.coords)
        geom_ee = ee.Geometry.Polygon(coords)
        stats   = ee.Image("USGS/SRTMGL1_003").reduceRegion(
            reducer=ee.Reducer.mean(), geometry=geom_ee,
            scale=500, maxPixels=1e9
        ).getInfo()
        alt_districts[dist] = float(stats.get('elevation', 1200))
    except:
        alt_districts[dist] = 1200.0
print(f" ✓")
for dist, alt in alt_districts.items():
    print(f"   {dist:<25} : {alt:.0f} m")

# AWC par district
print("   AWC sols...", end='', flush=True)
soil_awc_dist = {}
try:
    gdf_sols = gpd.read_file(SHAPEFILE_SOLS)
    if gdf_sols.crs is None:          gdf_sols = gdf_sols.set_crs("EPSG:4326")
    elif gdf_sols.crs.to_epsg() != 4326: gdf_sols = gdf_sols.to_crs("EPSG:4326")
    for _, row in gdf.iterrows():
        dist  = row[COL_DISTRICT]
        poly  = gpd.GeoDataFrame([row], geometry='geometry', crs=gdf.crs)
        inter = gpd.overlay(poly, gdf_sols[['geometry','FAOSOIL']],
                            how='intersection', keep_geom_type=False)
        if len(inter) > 0 and 'FAOSOIL' in inter.columns:
            inter['area'] = inter.geometry.area
            inter['awc']  = inter['FAOSOIL'].apply(get_awc)
            soil_awc_dist[dist] = int(round(
                (inter['awc']*inter['area']).sum()/inter['area'].sum()))
        else:
            soil_awc_dist[dist] = FAO_SOIL_AWC['default']
    del gdf_sols; gc.collect()
    print(f" ✓")
    for dist, awc in soil_awc_dist.items():
        print(f"   {dist:<25} : AWC = {awc} mm/m")
except Exception as e:
    print(f"  ({e}) → défaut 120 mm/m")
    for _, row in gdf.iterrows():
        soil_awc_dist[row[COL_DISTRICT]] = 120

In [ ]:
# ============================================================================
# 7. FORMULES SCIENTIFIQUES
# ============================================================================

def pression_atmo(z):
    return P0 * ((T0_K - L_ad * z) / T0_K) ** 5.26

def rh_scalaire(T_K, Td_K):
    T, Td = float(T_K)-273.15, float(Td_K)-273.15
    a, b  = 17.625, 243.04
    return float(np.clip(100*np.exp((a*Td)/(b+Td))/np.exp((a*T)/(b+T)), 0, 100))

def vpd_scalaire(Tmean_C, rh_pct):
    es = 0.6108 * np.exp(17.27*Tmean_C/(Tmean_C+237.3))
    return float(max(0.0, es*(1-rh_pct/100)))

def u2_scalaire(u10, v10):
    return float(np.sqrt(u10**2+v10**2)*(4.87/np.log(67.8*10-5.42)))

def dir_vent(u10, v10):
    return float(np.degrees(np.arctan2(float(u10), float(v10))) % 360)

def gdd_scalaire(Tmax, Tmin, culture):
    p = GDD_PARAMS[culture]
    return float(max(0.0, (min(Tmax,p['T_ceil'])+max(Tmin,p['T_base']))/2-p['T_base']))

def et0_scalaire(Tmax, Tmin, Tmean, rh, u2, Rs, alt):
    """Penman-Monteith FAO-56 avec altitude variable."""
    P     = pression_atmo(alt)
    gamma = 0.000665 * P
    delta = 4098*0.6108*np.exp(17.27*Tmean/(Tmean+237.3))/(Tmean+237.3)**2
    es    = (0.6108*np.exp(17.27*Tmax/(Tmax+237.3))
             +0.6108*np.exp(17.27*Tmin/(Tmin+237.3)))/2
    ea    = es*rh/100
    Rns   = (1-ALBEDO)*Rs
    Rs0   = max(Rs/0.75, 0.001)
    Rnl   = (SIGMA*((Tmax+273.16)**4+(Tmin+273.16)**4)/2
             *(0.34-0.14*np.sqrt(max(ea,0)))*(1.35*min(Rs/Rs0,1.0)-0.35))
    Rn    = Rns - Rnl
    num   = 0.408*delta*(Rn-G)+gamma*(900/(Tmean+273))*u2*(es-ea)
    return float(max(0.0, num/(delta+gamma*(1+0.34*u2))))

# ── Indices journaliers ──────────────────────────────────────────────────────

def cdd(serie):
    mx, cur = 0, 0
    for p in serie:
        cur = (cur+1) if p < SEUIL_PLUIE else 0
        mx  = max(mx, cur)
    return mx

def dsl_mean(serie):
    spells, cur = [], 0
    for p in serie:
        if p < SEUIL_PLUIE: cur += 1
        else:
            if cur > 0: spells.append(cur); cur = 0
    if cur > 0: spells.append(cur)
    return float(np.mean(spells)) if spells else 0.0

def extreme_metrics(serie):
    ext = [p for p in serie if p > SEUIL_EXTREME]
    tot = sum(serie)
    return {
        'extreme_days'    : len(ext),
        'extreme_total_mm': sum(ext),
        'extreme_pct'     : (sum(ext)/tot*100) if tot > 0 else 0.0,
    }

def stress_days_fn(serie_p, serie_et0, culture, phase):
    kc = KC_PARAMS[culture][phase]
    return int(sum(1 for p,e in zip(serie_p,serie_et0) if p < kc*e))

def water_deficit_fn(serie_p, serie_et0, culture, phase):
    kc = KC_PARAMS[culture][phase]
    return float(sum(max(0.0, kc*e-p) for p,e in zip(serie_p,serie_et0)))

# def wrsi_fn(total_p, total_et0, culture, awc_pm):
#     """Water Requirements Satisfaction Index — FAO-33."""
#     awc    = awc_pm * ROOT_DEPTH[culture] / 1000
#     raw    = DEPLETION_P[culture] * awc
#     kc_moy = np.mean([KC_PARAMS[culture][ph] for ph in PHASES])
#     etc    = kc_moy * total_et0
#     if etc <= 0: return 100.0
#     return float(np.clip((total_p+0.5*raw)/etc*100, 0, 100))

def wrsi_fn(phase_results_dist, total_et0, culture, awc_pm):
    """WRSI corrigé : ETC = sum(Kc_i * ET0_i) par phase."""
    awc = awc_pm * ROOT_DEPTH[culture] / 1000
    raw = DEPLETION_P[culture] * awc

    # Précip totale depuis les résultats par phase
    total_p = sum(                                          # ← ajout
        phase_results_dist[ph].get('precip_total_mm', 0)
        for ph in PHASES
    )
    # ETC pondéré par l'ET0 réelle de chaque phase
    etc = sum(
        KC_PARAMS[culture][ph] * phase_results_dist[ph].get('et0_mm', 0)
        for ph in PHASES
    )
    if etc <= 0: return 100.0
    return float(np.clip((total_p + 0.5 * raw) / etc * 100, 0, 100))


def onset_fn(serie):
    """Dunning et al. (2016)"""
    n = len(serie)
    for i in range(n-33):
        win = serie[i:i+3]
        if sum(win) < 20: continue
        if sum(1 for p in win if p >= SEUIL_PLUIE) < 2: continue
        suiv = serie[i+3:i+33]
        sec, mx = 0, 0
        for p in suiv:
            sec = (sec+1) if p < SEUIL_PLUIE else 0
            mx  = max(mx, sec)
        if mx <= 7: return float(i)
    return np.nan

def cessation_fn(serie, awc_mm):
    """Araya et al. (2010)"""
    stock = awc_mm; debut = len(serie)//2
    for i in range(debut, len(serie)):
        stock = min(awc_mm, max(0.0, stock+serie[i]-ET0_CESS))
        if stock < SEUIL_SW_CESS:
            consec, sw = 1, stock
            for j in range(i+1, min(i+DUR_CESS, len(serie))):
                sw = min(awc_mm, max(0.0, sw+serie[j]-ET0_CESS))
                if sw < SEUIL_SW_CESS: consec += 1
                else: break
            if consec >= DUR_CESS: return float(i)
    return np.nan

In [ ]:
# 8. UTILITAIRES DATES & BLOCS
# ============================================================================

def nb_jours_mois(an, mois):
    if mois in [1,3,5,7,8,10,12]: return 31
    if mois in [4,6,9,11]:        return 30
    return 29 if (an%4==0 and (an%100!=0 or an%400==0)) else 28

def dates_phase(annee, phase):
    dates = []
    for mois, _, decal in PHASES_MOIS[phase]:
        an = annee + decal
        for j in range(1, nb_jours_mois(an, mois)+1):
            dates.append(date(an, mois, j))
    return sorted(dates)

def decouper_en_blocs(liste_dates, taille=10):
    return [liste_dates[i:i+taille] for i in range(0, len(liste_dates), taille)]

In [ ]:
#==============================================================================
# 9. TÉLÉCHARGEMENT ERA5-LAND (MONTHLY_AGGR — collection à jour)
# ============================================================================

def _get_time_dim(ds):
    for c in ['time', 'valid_time']:
        if c in ds.dims or c in ds.coords:
            return c
    raise KeyError("Dimension temporelle introuvable")

def _decouper_district(ds, geom):
    b       = geom.bounds
    lat_dim = 'latitude' if 'latitude' in ds.dims else 'lat'
    lon_dim = 'longitude' if 'longitude' in ds.dims else 'lon'
    return ds.sel({lon_dim: slice(b[0], b[2]),
                   lat_dim: slice(b[3], b[1])}), lat_dim, lon_dim

def _concat_blocs_safe(datasets, t_dim):
    if len(datasets) == 1:
        return datasets[0]
    ref = datasets[0]
    spatial_dims = [d for d in ref.dims if d != t_dim]
    coords_communs = {}
    for dim in spatial_dims:
        c = set(ref[dim].values.tolist())
        for ds in datasets[1:]:
            if dim in ds.dims:
                c &= set(ds[dim].values.tolist())
        coords_communs[dim] = sorted(c, reverse=(dim in ['latitude','lat']))
    return xr.concat([ds.sel(coords_communs) for ds in datasets], dim=t_dim)

def telecharger_era5_bloc(dates_bloc):
    if not dates_bloc: return []
    from collections import defaultdict
    groupes = defaultdict(list)
    for d in dates_bloc:
        groupes[(d.year, d.month)].append(f"{d.day:02d}")
    fichiers_nc = []
    for (an, mois), jours in groupes.items():
        ms  = f"{mois:02d}"
        tag = f"{an}{ms}_j{''.join(jours)}"
        out = os.path.join(TEMP_DIR, f"era5_{tag}.nc")
        if os.path.exists(out) and os.path.getsize(out) > 0:
            fichiers_nc.append(out); continue
        tmp = out + ".dl"
        for tentative in range(1, MAX_RETRY+1):
            try:
                cds.retrieve('reanalysis-era5-land', {
                    'variable': [
                        '2m_temperature', '2m_dewpoint_temperature',
                        '10m_u_component_of_wind', '10m_v_component_of_wind',
                        'surface_solar_radiation_downwards',
                        'volumetric_soil_water_layer_1',
                    ],
                    'year': str(an), 'month': ms, 'day': jours,
                    'time': [f'{h:02d}:00' for h in range(24)],
                    'area': bbox_cds, 'format': 'netcdf',
                }, tmp)
                if zipfile.is_zipfile(tmp):
                    with zipfile.ZipFile(tmp) as z:
                        z.extractall(TEMP_DIR)
                    os.remove(tmp)
                    for n in z.namelist():
                        if n.endswith('.nc'):
                            os.rename(os.path.join(TEMP_DIR, n), out)
                            break
                else:
                    os.rename(tmp, out)
                fichiers_nc.append(out)
                time.sleep(DELAI_ERA5_S)
                break
            except Exception as e:
                if os.path.exists(tmp): os.remove(tmp)
                if tentative < MAX_RETRY:
                    time.sleep(DELAI_ERA5_S * tentative)
    return fichiers_nc

def extraire_era5_journalier(fichiers_nc, geom, alt):
    """
    Extrait les variables journalières ERA5-Land.
    Alt est maintenant passé en argument (variable par district).
    """
    fichiers_valides = [f for f in (fichiers_nc or [])
                        if f and os.path.exists(f)]
    if not fichiers_valides: return pd.DataFrame()

    ds_open = []
    rows    = []
    try:
        blocs_3d, t_dim_ref = [], None
        for f in fichiers_valides:
            ds = xr.open_dataset(f); ds_open.append(ds)
            try:
                ds_c, _, _ = _decouper_district(ds, geom)
                if t_dim_ref is None:
                    t_dim_ref = _get_time_dim(ds_c)
                blocs_3d.append(ds_c)
            except: pass

        if not blocs_3d: return pd.DataFrame()
        ds_b  = _concat_blocs_safe(blocs_3d, t_dim_ref)
        t_dim = _get_time_dim(ds_b)
        times = pd.DatetimeIndex(ds_b[t_dim].values)
        days  = np.unique(times.date)

        for day in days:
            idx = np.where(times.date == day)[0]
            if not len(idx): continue
            row = {'date': pd.Timestamp(day)}

            if 't2m' in ds_b:
                t2m = ds_b['t2m'].isel({t_dim: idx}).values - 273.15
                row['tmax_C']  = float(np.nanmax(t2m))
                row['tmin_C']  = float(np.nanmin(t2m))
                row['tmean_C'] = float(np.nanmean(t2m))
                row['dtr_C']   = row['tmax_C'] - row['tmin_C']
                row['thermal_stress'] = int(np.any(t2m > SEUIL_TSTRESS))
                row['hot_night']      = int(row['tmin_C'] > SEUIL_HOT_N)
                for culture in GDD_PARAMS:
                    row[f'gdd_{culture.lower()}'] = gdd_scalaire(
                        row['tmax_C'], row['tmin_C'], culture)

            if 't2m' in ds_b and 'd2m' in ds_b:
                t_m = float(np.nanmean(ds_b['t2m'].isel({t_dim: idx}).values))
                d_m = float(np.nanmean(ds_b['d2m'].isel({t_dim: idx}).values))
                row['rh_pct'] = rh_scalaire(t_m, d_m)

            if 'tmean_C' in row and 'rh_pct' in row:
                row['vpd_kPa'] = vpd_scalaire(row['tmean_C'], row['rh_pct'])

            if 'swvl1' in ds_b:
                row['soil_moisture'] = float(
                    np.nanmean(ds_b['swvl1'].isel({t_dim: idx}).values))

            if 'u10' in ds_b and 'v10' in ds_b:
                u = float(np.nanmean(ds_b['u10'].isel({t_dim: idx}).values))
                v = float(np.nanmean(ds_b['v10'].isel({t_dim: idx}).values))
                row['wind_speed_2m'] = u2_scalaire(u, v)
                row['wind_dir_deg']  = dir_vent(u, v)

            if 'ssrd' in ds_b:
                ssrd_m = np.array([
                    float(np.nanmean(ds_b['ssrd'].isel({t_dim: [i]}).values))
                    for i in idx])
                deltas = np.diff(np.concatenate([[0.0], ssrd_m]))
                neg    = np.where(deltas < 0)[0]
                deltas[neg] = ssrd_m[neg]
                row['solar_rad_MJm2d'] = float(np.sum(deltas)) / 1e6

            req = ['tmax_C','tmin_C','tmean_C','rh_pct',
                   'wind_speed_2m','solar_rad_MJm2d']
            if all(k in row for k in req):
                row['et0_mm'] = et0_scalaire(
                    row['tmax_C'], row['tmin_C'], row['tmean_C'],
                    row['rh_pct'], row['wind_speed_2m'],
                    row['solar_rad_MJm2d'],
                    alt   # ← altitude variable par district
                )
            rows.append(row)

        return pd.DataFrame(rows)
    finally:
        for ds in ds_open:
            try: ds.close()
            except: pass
        gc.collect()

In [ ]:
# ============================================================================
# 10. CHIRPS
# ============================================================================

def telecharger_chirps_jour(an, mois, jour):
    ms, js = f"{mois:02d}", f"{jour:02d}"
    fname  = f"chirps-v2.0.{an}.{ms}.{js}.tif"
    local  = os.path.join(TEMP_DIR, fname)
    if os.path.exists(local) and os.path.getsize(local) > 0:
        return local
    url = (f"https://data.chc.ucsb.edu/products/CHIRPS-2.0/"
           f"global_daily/tifs/p05/{an}/{fname}.gz")
    for tentative in range(1, MAX_RETRY+1):
        try:
            r = requests.get(url, stream=True, timeout=90)
            if r.status_code != 200: return None
            gz = local + ".gz"
            with open(gz,'wb') as f_:
                for chunk in r.iter_content(8192): f_.write(chunk)
            with gzip.open(gz,'rb') as fi, open(local,'wb') as fo:
                shutil.copyfileobj(fi, fo)
            os.remove(gz)
            time.sleep(DELAI_CHIRPS_S)
            return local
        except:
            if tentative < MAX_RETRY:
                time.sleep(DELAI_CHIRPS_S * tentative)
    return None

def extraire_precip_jour(fichier_tif, geom):
    if not fichier_tif or not os.path.exists(fichier_tif): return 0.0
    try:
        with rasterio.open(fichier_tif) as src:
            out, _ = rasterio_mask(src, [geom], crop=True,
                                   all_touched=True, nodata=-9999)
            vals = out[0].astype(float)
            valid = vals[(vals > -9000) & (vals < 9999)]
            return float(valid.mean()) if len(valid) > 0 else 0.0
    except: return 0.0

In [ ]:
# ============================================================================
# 11. LANDSAT / VCI
# ============================================================================

def extraire_landsat(geom, annee, phase):
    dates_p = {
        'P1': (f'{annee-1}-11-01', f'{annee-1}-12-31'),
        'P2': (f'{annee}-01-01',   f'{annee}-02-28'),
        'P3': (f'{annee}-03-01',   f'{annee}-05-31'),
    }
    d1, d2 = dates_p[phase]
    try:
        ge = ee.Geometry.Polygon(list(geom.exterior.coords))
        def mask_clouds(img):
            qa = img.select('QA_PIXEL')
            return img.updateMask(
                qa.bitwiseAnd(1<<3).neq(0).Or(qa.bitwiseAnd(1<<4).neq(0)).Not())
        def add_idx(nir, red, green):
            def _inner(img):
                return img.addBands([
                    img.normalizedDifference([nir, red]).rename('NDVI'),
                    img.normalizedDifference([green, nir]).rename('NDWI'),
                ])
            return _inner
        missions = [
            ('LANDSAT/LT05/C02/T1_L2','SR_B4','SR_B3','SR_B2'),
            ('LANDSAT/LE07/C02/T1_L2','SR_B4','SR_B3','SR_B2'),
            ('LANDSAT/LC08/C02/T1_L2','SR_B5','SR_B4','SR_B3'),
            ('LANDSAT/LC09/C02/T1_L2','SR_B5','SR_B4','SR_B3'),
        ]
        merged = None
        for name, nir, red, green in missions:
            col = (ee.ImageCollection(name).filterBounds(ge)
                   .filterDate(d1, d2).map(mask_clouds)
                   .map(add_idx(nir, red, green)))
            merged = col if merged is None else merged.merge(col)
        if merged is None or merged.size().getInfo() == 0:
            return {'ndvi': np.nan, 'ndwi': np.nan}
        comp  = merged.select(['NDVI','NDWI']).median()
        stats = comp.reduceRegion(ee.Reducer.mean(), ge, 30, maxPixels=1e9).getInfo()
        return {'ndvi': stats.get('NDVI', np.nan),
                'ndwi': stats.get('NDWI', np.nan)}
    except:
        return {'ndvi': np.nan, 'ndwi': np.nan}

def calculer_vci(df, phases):
    """Kogan (1995)"""
    df = df.copy()
    for phase in phases:
        cn, cv = f'{phase}_ndvi', f'{phase}_vci'
        if cn not in df.columns: continue
        for dist in df['district'].unique():
            m = df['district'] == dist
            s = df.loc[m, cn].dropna()
            if len(s) < 2:
                df.loc[m, cv] = np.nan; continue
            mn, mx = s.min(), s.max()
            df.loc[m, cv] = (0.0 if mn == mx
                             else (df.loc[m, cn]-mn)/(mx-mn)*100.0)
    return df

In [ ]:
# ============================================================================
# 12. AGRÉGATION PAR PHASE
# ============================================================================

def agreger_phase(serie_p, serie_et0, df_era5, phase, awc_pm, alt):
    """
    Agrège toutes les variables pour une phase.
    Note : SPI/SPEI supprimés — intégrés via jointure finale.
    """
    r = {}

    # Précipitations
    r['precip_total_mm']      = float(sum(serie_p))
    r['rainy_days']           = int(sum(1 for p in serie_p if p >= SEUIL_PLUIE))
    r['precip_intensity_mm']  = (r['precip_total_mm']/r['rainy_days']
                                 if r['rainy_days'] > 0 else 0.0)
    r['cdd']                  = cdd(serie_p)
    r['dry_spell_mean']       = dsl_mean(serie_p)
    r.update(extreme_metrics(serie_p))

    # ERA5 agrégation
    if len(df_era5) > 0:
        r['tmax_C']  = float(df_era5['tmax_C'].max())
        r['tmin_C']  = float(df_era5['tmin_C'].min())
        r['tmean_C'] = float(df_era5['tmean_C'].mean())
        r['dtr_C']   = float(df_era5['dtr_C'].mean())
        r['thermal_stress_days'] = int(df_era5['thermal_stress'].sum())
        r['hot_nights']          = int(df_era5['hot_night'].sum())

        for culture in GDD_PARAMS:
            c = culture.lower()
            if f'gdd_{c}' in df_era5:
                r[f'gdd_{c}'] = float(df_era5[f'gdd_{c}'].sum())

        for col in ['rh_pct','vpd_kPa','soil_moisture',
                    'wind_speed_2m','wind_dir_deg']:
            if col in df_era5:
                r[col] = float(df_era5[col].mean())

        if 'et0_mm' in df_era5 and len(serie_et0) > 0:
            r['et0_mm'] = float(sum(serie_et0))
            for culture in KC_PARAMS:
                c  = culture.lower()
                kc = KC_PARAMS[culture][phase]
                r[f'etc_{c}_mm']          = kc * r['et0_mm']
                r[f'water_deficit_{c}_mm'] = water_deficit_fn(
                    serie_p, serie_et0, culture, phase)
                r[f'stress_days_{c}']     = stress_days_fn(
                    serie_p, serie_et0, culture, phase)
    return r

In [ ]:
# ============================================================================
# 13. BOUCLE PRINCIPALE
# ============================================================================

print("\n5. BOUCLE PRINCIPALE")
print("="*80)

all_rows = []

for annee in ANNEES:
    print(f"\n{'='*60}")
    print(f"  CAMPAGNE {annee-1}/{annee}")
    print(f"{'='*60}")

    precip_saison = {d: [] for d in districts}
    et0_saison    = {d: [] for d in districts}
    phase_results = {d: {} for d in districts}

    for phase in PHASES:
        print(f"\n  ── Phase {phase} ──")
        phase_dates = dates_phase(annee, phase)
        blocs       = decouper_en_blocs(phase_dates, BLOC_ERA5_JOURS)
        print(f"     {len(phase_dates)} jours | {len(blocs)} blocs")

        series = {d: {'precip': [], 'et0': [], 'era5_rows': []}
                  for d in districts}

        for i_bloc, bloc in enumerate(blocs):
            print(f"     Bloc {i_bloc+1}/{len(blocs)} "
                  f"({bloc[0]} → {bloc[-1]})...", end=' ')
            fichiers_nc = telecharger_era5_bloc(bloc)

            for _, row_d in gdf.iterrows():
                dist = row_d[COL_DISTRICT]
                alt  = alt_districts[dist]  # altitude variable
                df_j = (extraire_era5_journalier(fichiers_nc, row_d.geometry, alt)
                        if fichiers_nc else pd.DataFrame())
                if len(df_j) > 0 and 'et0_mm' in df_j.columns:
                    series[dist]['et0'].extend(df_j['et0_mm'].tolist())
                    series[dist]['era5_rows'].append(df_j)

            for d in bloc:
                f_tif = telecharger_chirps_jour(d.year, d.month, d.day)
                for _, row_d in gdf.iterrows():
                    dist = row_d[COL_DISTRICT]
                    series[dist]['precip'].append(
                        extraire_precip_jour(f_tif, row_d.geometry))
                if f_tif and os.path.exists(f_tif):
                    try: os.remove(f_tif)
                    except: pass

            for f in (fichiers_nc or []):
                if f and os.path.exists(f):
                    try: os.remove(f)
                    except: pass

            print("✓")
            gc.collect()

        print(f"     Agrégation...", end=' ')
        for dist in districts:
            alt    = alt_districts[dist]
            awc_pm = soil_awc_dist[dist]
            sp     = series[dist]['precip']
            se     = series[dist]['et0']

            df_era5_phase = (
                pd.concat(series[dist]['era5_rows'], ignore_index=True)
                if series[dist]['era5_rows'] else pd.DataFrame())

            precip_saison[dist].extend(sp)
            et0_saison[dist].extend(se)

            phase_results[dist][phase] = agreger_phase(
                sp, se, df_era5_phase, phase, awc_pm, alt)

            if phase in ('P2', 'P3'):
                geom_dist = gdf.loc[gdf[COL_DISTRICT]==dist,'geometry'].iloc[0]
                ls = extraire_landsat(geom_dist, annee, phase)
                phase_results[dist][phase]['ndvi'] = ls['ndvi']
                phase_results[dist][phase]['ndwi'] = ls['ndwi']

        del series; gc.collect()
        print("✓")

    # ── Indices saisonniers ────────────────────────────────────────────────
    print(f"\n  Indices saisonniers...", end=' ')
    for dist in districts:
        awc_pm  = soil_awc_dist[dist]
        serie_p = precip_saison[dist]
        serie_e = et0_saison[dist]

        # Onset & Cessation
        onset_d = onset_fn(serie_p)
        awc_riz = awc_pm * ROOT_DEPTH['Rice'] / 1000
        cess_d  = cessation_fn(serie_p, awc_riz)
        sl = (float(cess_d - onset_d)
              if not np.isnan(onset_d) and not np.isnan(cess_d) else np.nan)

        # WRSI
        # total_p  = sum(phase_results[dist][ph].get('precip_total_mm', 0) for ph in PHASES)
        # total_et = sum(phase_results[dist][ph].get('et0_mm', 0) for ph in PHASES)
        # wrsi_vals = {c: wrsi_fn(total_p, total_et, c, awc_pm) for c in GDD_PARAMS}
        # APRÈS
        total_et = sum(phase_results[dist][ph].get('et0_mm', 0) for ph in PHASES)
        wrsi_vals = {c: wrsi_fn(phase_results[dist], c, awc_pm) for c in GDD_PARAMS}

        # ── SPEI précalculés : lecture directe ──────────────────────────
        # annee - 1 = année de début de campagne
        # ex: annee=2013 → campagne 2012/13 → annee_campagne=2012
        spei_vals = get_spei_district_annee(dist, int(annee - 1))  # cast explicite

        row_out = {
            'annee'               : annee,
            'district'            : dist,
            'onset_day'           : onset_d,
            'cessation_day'       : cess_d,
            'season_length_days'  : sl,
            # SPEI précalculés (lus depuis fichier, pas recalculés)
            'spei3_fev'           : spei_vals['spei3_fev'],
            'spei3_fev_cat'       : spei_vals['spei3_fev_cat'],
            'spei6_mai'           : spei_vals['spei6_mai'],
            'spei6_mai_cat'       : spei_vals['spei6_mai_cat'],
            'spei12_mai'          : spei_vals['spei12_mai'],
            'spei12_mai_cat'      : spei_vals['spei12_mai_cat'],
            **{f'wrsi_{c.lower()}': v for c, v in wrsi_vals.items()},
        }

        for phase in PHASES:
            for var, val in phase_results[dist][phase].items():
                row_out[f'{phase}_{var}'] = val

        all_rows.append(row_out)

    # Sauvegarde annuelle
    df_ann = pd.DataFrame(all_rows[-len(districts):])
    df_ann.to_csv(
        f"{OUTPUT_DIR}/02_PROCESSED/ANNUAL/phase_{annee}.csv", index=False)

    del precip_saison, et0_saison, phase_results
    for f in os.listdir(TEMP_DIR):
        try: os.remove(os.path.join(TEMP_DIR, f))
        except: pass
    gc.collect()
    print(f"✓  → phase_{annee}.csv")

In [ ]:
# ============================================================================
# 14. VCI + FUSION RENDEMENTS + SAUVEGARDE FINALE
# ============================================================================

print("\n6. VCI + FUSION RENDEMENTS")
print("-"*40)

df_final = pd.DataFrame(all_rows)
df_final = calculer_vci(df_final, ['P2', 'P3'])

# Fusion rendements
df_yield = pd.read_excel(EXCEL_RENDEMENTS)
df_yield.columns = [c.strip().lower() for c in df_yield.columns]
for col in ['année','annee','year']:
    if col in df_yield.columns:
        df_yield.rename(columns={col: 'annee'}, inplace=True); break

df_merged = pd.merge(df_final, df_yield, on=['annee','district'], how='inner')

# Sauvegarde
out_csv  = f"{OUTPUT_DIR}/03_FINAL/dataset_complet.csv"
out_xlsx = f"{OUTPUT_DIR}/03_FINAL/dataset_complet.xlsx"
df_merged.to_csv(out_csv,  index=False, encoding='utf-8-sig')
df_merged.to_excel(out_xlsx, index=False)

# Métadonnées
with open(f"{OUTPUT_DIR}/03_FINAL/metadata.txt", 'w') as f:
    f.write("PIPELINE AGROCLIMATIQUE — HAUTE MATSIATRA [v2]\n")
    f.write("="*60 + "\n")
    f.write(f"Date         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Période      : {min(ANNEES)}–{max(ANNEES)}\n")
    f.write(f"Campagnes    : {min(ANNEES)-1}/{min(ANNEES)} → {max(ANNEES)-1}/{max(ANNEES)}\n")
    f.write(f"Districts    : {', '.join(districts)}\n")
    f.write(f"Obs.         : {df_merged.shape[0]}\n")
    f.write(f"Variables    : {df_merged.shape[1]}\n\n")
    f.write("SOURCES DONNÉES :\n")
    f.write(f"  Précipitations  : CHIRPS v2 (~5 km, journalier)\n")
    f.write(f"  Météo/ET0       : ERA5-Land CDS (journalier, FAO-56 PM)\n")
    f.write(f"  Altitude ET0    : SRTM 30m via GEE (variable par district)\n")
    f.write(f"  Végétation      : Landsat 5/7/8/9 (NDVI, NDWI, VCI)\n")
    f.write(f"  SPEI précalculé : {SPEI_PKL}\n")
    f.write(f"    → SPEI-3 fév  (phase critique Déc–Jan–Fév)\n")
    f.write(f"    → SPEI-6 mai  (bilan saison Déc–Mai)\n")
    f.write(f"    → SPEI-12 mai (fond hydrologique)\n")
    f.write(f"    → Baseline    : 1991–2020 (norme OMM)\n")
    f.write(f"    → Méthode     : Vicente-Serrano et al. (2010)\n")
    f.write(f"  Rendements      : {EXCEL_RENDEMENTS}\n\n")
    f.write("COLONNES SPEI DANS LE DATASET FINAL :\n")
    for col in ['spei3_fev','spei3_fev_cat','spei6_mai',
                'spei6_mai_cat','spei12_mai','spei12_mai_cat']:
        if col in df_merged.columns:
            n_non_nan = df_merged[col].notna().sum()
            f.write(f"  {col:<20} : {n_non_nan}/{len(df_merged)} valeurs renseignées\n")

print(f"\n{'='*80}")
print(f"PIPELINE TERMINÉ")
print(f"Dataset final : {out_csv}")
print(f"{df_merged.shape[0]} obs × {df_merged.shape[1]} variables")
print(f"\n  Colonnes SPEI dans le dataset :")
for col in ['spei3_fev','spei3_fev_cat','spei6_mai',
            'spei6_mai_cat','spei12_mai','spei12_mai_cat']:
    if col in df_merged.columns:
        n = df_merged[col].notna().sum()
        print(f"    {col:<22}: {n}/{len(df_merged)} valeurs")
print(f"\n   Fin : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*80}")